# Bronze Batch Finalizer

Marks batch ingestion completion and updates watermarks.

**Purpose**: Track batch completion and data freshness  
**Pattern**: Immutable batch state records  
**Watermarks**: Enable incremental processing downstream

In [0]:
# Batch finalization parameters
dbutils.widgets.text("batch_id", "", "Batch ID")
dbutils.widgets.text("source_name", "", "Source Name")
dbutils.widgets.text("records_ingested", "0", "Records Ingested")
dbutils.widgets.text("records_failed", "0", "Records Failed")
dbutils.widgets.text("watermark_value", "", "Watermark Value")
dbutils.widgets.dropdown("status", "success", ["success", "partial", "failed"], "Status")
dbutils.widgets.text("catalog", "main", "Catalog")
dbutils.widgets.text("schema", "bronze", "Schema")

# Get values
batch_id = dbutils.widgets.get("batch_id")
source_name = dbutils.widgets.get("source_name")
records_ingested = int(dbutils.widgets.get("records_ingested"))
records_failed = int(dbutils.widgets.get("records_failed"))
watermark_value = dbutils.widgets.get("watermark_value")
status = dbutils.widgets.get("status")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

if not batch_id:
    raise ValueError("batch_id is required")
if not source_name:
    raise ValueError("source_name is required")

In [0]:
%sql
-- Create batch metadata table
CREATE TABLE IF NOT EXISTS ${catalog}.${schema}.batch_metadata (
  batch_id STRING NOT NULL,
  source_name STRING NOT NULL,
  records_ingested INT,
  records_failed INT,
  watermark_value STRING,
  status STRING NOT NULL,
  batch_start_timestamp TIMESTAMP,
  batch_end_timestamp TIMESTAMP NOT NULL,
  finalization_timestamp TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Immutable batch execution records and watermarks'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true'
);

In [0]:
from pyspark.sql import functions as F

# Create batch metadata record
batch_df = spark.createDataFrame([(
    batch_id,
    source_name,
    records_ingested,
    records_failed,
    watermark_value if watermark_value else None,
    status,
    F.current_timestamp(),
    F.current_timestamp(),
    F.current_timestamp()
)], schema="""
    batch_id STRING,
    source_name STRING,
    records_ingested INT,
    records_failed INT,
    watermark_value STRING,
    status STRING,
    batch_start_timestamp TIMESTAMP,
    batch_end_timestamp TIMESTAMP,
    finalization_timestamp TIMESTAMP
""")

# Append to Bronze table
target_table = f"{catalog}.{schema}.batch_metadata"
batch_df.write.mode("append").saveAsTable(target_table)

print(f"✓ Batch finalized in {target_table}")
print(f"  Batch ID: {batch_id}")
print(f"  Source: {source_name}")
print(f"  Status: {status}")
print(f"  Records ingested: {records_ingested}")
print(f"  Records failed: {records_failed}")
if watermark_value:
    print(f"  Watermark: {watermark_value}")

In [0]:
%sql
-- Create watermark tracking table
CREATE TABLE IF NOT EXISTS ${catalog}.${schema}.watermarks (
  source_name STRING NOT NULL,
  watermark_value STRING NOT NULL,
  last_batch_id STRING,
  update_timestamp TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Current watermark state per source'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true'
);

In [0]:
# Update watermark if provided
if watermark_value and status == "success":
    from delta.tables import DeltaTable
    
    watermark_table = f"{catalog}.{schema}.watermarks"
    
    # Create watermark update record
    watermark_df = spark.createDataFrame([(
        source_name,
        watermark_value,
        batch_id,
        F.current_timestamp()
    )], schema="source_name STRING, watermark_value STRING, last_batch_id STRING, update_timestamp TIMESTAMP")
    
    # Check if table exists
    if spark.catalog.tableExists(watermark_table):
        # Merge watermark
        delta_table = DeltaTable.forName(spark, watermark_table)
        delta_table.alias("target").merge(
            watermark_df.alias("source"),
            "target.source_name = source.source_name"
        ).whenMatchedUpdate(
            set={
                "watermark_value": "source.watermark_value",
                "last_batch_id": "source.last_batch_id",
                "update_timestamp": "source.update_timestamp"
            }
        ).whenNotMatchedInsertAll().execute()
    else:
        # First time - insert
        watermark_df.write.mode("append").saveAsTable(watermark_table)
    
    print(f"✓ Watermark updated in {watermark_table}")
    print(f"  Source: {source_name}")
    print(f"  Watermark: {watermark_value}")